# Portfolio Efficient Frontier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frankhuettner/ManSciDA-Excel-to-app/blob/main/portfolio-app/portfolio.ipynb)

**Model:** Quadratic Program (QP)

$$\min_{w} \quad w^\top \Sigma w$$
$$\text{s.t.} \quad \sum_i w_i = 1, \quad r^\top w \geq r^*, \quad w_{\text{risky}} \geq 0$$

where $\Sigma$ is the covariance matrix and $r^*$ is the target return.

**Solver:** CVXPY + Clarabel  
**Same code** runs in the browser app (`portfolio-app/index.html`) via Pyodide.

---
### Environment
| Context | Runtime | Package name | Install |
|---|---|---|---|
| This notebook | Google Colab / local Python (uv) | `cvxpy` | first cell below / `uv sync` |
| Browser app | Pyodide (WebAssembly) | `cvxpy-base` | `pyodide.loadPackage(["cvxpy-base", "clarabel"])` |

The solver **code is identical** in both contexts — only the packaging differs.

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
import pandas as pd

print(f"cvxpy  {cp.__version__}")
print(f"numpy  {np.__version__}")

In [ ]:
# Install packages — works in Google Colab and locally
# (In Colab: Runtime → Run all, or run this cell first)
import sys
if "google.colab" in sys.modules:
    %pip install cvxpy clarabel -q

## Data
Source: `portfolio-efficient-frontier.xlsx`, sheet `(e)`

In [ ]:
NAMES = ["A", "B", "C", "F"]

r   = np.array([0.12, 0.09, 0.05, 0.03])   # expected returns
sig = np.array([0.22, 0.14, 0.08, 0.00])   # std deviations (F is risk-free)

# Correlation matrix for the three risky assets A, B, C
corr = np.array([[1.0, 0.5, 0.2],
                 [0.5, 1.0, 0.5],
                 [0.2, 0.5, 1.0]])

n_r = 3   # risky assets
n   = 4   # total assets (incl. risk-free F)
rf  = float(r[-1])

# Build full 4x4 covariance matrix (F row/col stays zero)
Sigma = np.zeros((n, n))
for i in range(n_r):
    for j in range(n_r):
        Sigma[i, j] = corr[i, j] * sig[i] * sig[j]

pd.DataFrame(Sigma, index=NAMES, columns=NAMES).round(6)

## Single portfolio — minimize variance for a target return

This is **identical** to the `_qp()` function embedded in `index.html`.

In [ ]:
def solve_qp(r_target, no_short_risky=True, allow_borrow=True):
    """Minimize portfolio variance subject to a target return."""
    w = cp.Variable(n)
    cons = [cp.sum(w) == 1, r @ w >= r_target]
    if no_short_risky:
        cons.append(w[:n_r] >= 0)      # no shorting A, B, C
    if not allow_borrow:
        cons.append(w[n_r:] >= 0)      # no borrowing at risk-free
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), cons)
    prob.solve(solver=cp.CLARABEL, verbose=False)
    if prob.status not in ("optimal", "optimal_inaccurate") or w.value is None:
        return None
    wv  = w.value
    ret = float(r @ wv)
    std = float(np.sqrt(max(wv @ Sigma @ wv, 0.0)))
    return wv, ret, std

### Test — reproduce the Excel solution at 13% target return

In [ ]:
wv, ret, std = solve_qp(0.13)
shp = (ret - rf) / std

result = pd.Series(
    {"Return": f"{ret*100:.2f}%",
     "Std Dev": f"{std*100:.2f}%",
     "Sharpe": f"{shp:.4f}",
     **{f"w_{nm}": f"{wv[i]:.4f}" for i, nm in enumerate(NAMES)}}
)
display(result)

# Assertions against Excel values
assert abs(ret - 0.13)       < 1e-4, f"Return mismatch: {ret}"
assert abs(std - 0.2052)     < 1e-3, f"Std Dev mismatch: {std}"
assert abs(wv[0] - 0.5055)   < 1e-3, "w_A mismatch"
assert abs(wv[3] - (-0.6428))< 1e-3, "w_F (borrowing) mismatch"

print("✓ Matches Excel solution")

## Efficient frontier

Sweep `r_target` from just above the risk-free rate to above the maximum risky return (leverage allowed).

> **Theory check:** with a risk-free asset and no short-selling constraint on risky assets,
> the frontier should be a **straight line** (Capital Market Line) and the **Sharpe ratio
> should be constant** along it — this is two-fund separation.

In [ ]:
targets = np.linspace(rf + 0.001, max(r[:n_r]) * 1.35, 40)

rows = []
for rt in targets:
    res = solve_qp(rt)
    if res is not None:
        wv_f, ret_f, std_f = res
        shp_f = (ret_f - rf) / std_f if std_f > 1e-9 else 0.0
        rows.append({"r_target": rt, "return": ret_f, "std": std_f, "sharpe": shp_f,
                     **{f"w_{nm}": wv_f[i] for i, nm in enumerate(NAMES)}})

frontier = pd.DataFrame(rows)

print(f"Feasible points: {len(frontier)}")
print(f"Sharpe range: {frontier['sharpe'].min():.4f} – {frontier['sharpe'].max():.4f}  (constant = CML confirmed)")
frontier[["return", "std", "sharpe"]].mul([100, 100, 1]).round(3).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Efficient frontier
ax.plot(frontier["std"] * 100, frontier["return"] * 100,
        color="#7c3aed", lw=2, label="Efficient frontier")

# Individual assets
for i, nm in enumerate(NAMES):
    ax.scatter(sig[i] * 100, r[i] * 100, zorder=5,
               s=80, marker="^", color="#f59e0b",
               label=nm if i == 0 else None)
    ax.annotate(nm, (sig[i] * 100 + 0.3, r[i] * 100), fontsize=10)

# Min-variance portfolio
mv = frontier.loc[frontier["std"].idxmin()]
ax.scatter(mv["std"] * 100, mv["return"] * 100,
           s=120, marker="D", color="#10b981", zorder=6, label="Min variance")

# Target 13% portfolio
tgt = solve_qp(0.13)
if tgt:
    _, t_ret, t_std = tgt
    ax.scatter(t_std * 100, t_ret * 100,
               s=120, marker="o", color="#ef4444", zorder=6, label="Target 13%")

ax.set_xlabel("Standard deviation (%)", fontsize=11)
ax.set_ylabel("Expected return (%)", fontsize=11)
ax.set_title("Efficient Frontier (no short-selling, borrowing allowed)", fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key portfolios

In [ ]:
i_mv  = frontier["std"].idxmin()
i_ms  = frontier["sharpe"].idxmax()

key = frontier.loc[[i_mv, i_ms]].copy()
key.index = ["Min Variance", "Max Sharpe"]

display_cols = ["return", "std", "sharpe"] + [f"w_{nm}" for nm in NAMES]
key[display_cols].mul(
    [100, 100, 1] + [1]*n
).round(3).rename(columns={"return": "Return %", "std": "Std %", "sharpe": "Sharpe"})